# Player Number-Guided AI Highlight Video Curator
**CSCI 5922 — Mateusz Muszynski & Colin Wallace**

Tracks jersey **#19** through *Denver vs. Virginia | 2025 ACC Men's Soccer* and assembles a personal highlight reel.

### Before first run
1. `Runtime → Change runtime type → T4 GPU`
2. Upload the match video to Google Drive (e.g. `My Drive/soccer/`)
3. Run cells top-to-bottom

### Re-running
`Runtime → Run all` — every cell is idempotent and skips work that is already done.

---
## 0 — GPU check

In [ ]:
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU — go to Runtime → Change runtime type → T4 GPU, then re-run all")

---
## 1 — Clone / update repo & install dependencies

In [ ]:
import os, sys

REPO_URL = "https://github.com/mateusz-muszynski/csci5922-highlight-curator.git"  # ← your repo
REPO_DIR = "/content/csci5922"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Pull latest fixes on every re-run
    !cd {REPO_DIR} && git pull --rebase

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Repo: {os.getcwd()}")
!git log --oneline -5

In [ ]:
!pip install -q \
    ultralytics>=8.2.0 \
    supervision>=0.21.0 \
    opencv-python-headless \
    albumentations \
    SoccerNet \
    ffmpeg-python \
    imageio[ffmpeg] \
    PyYAML tqdm rich

import torch, torchvision, ultralytics, supervision, cv2
print(f"torch {torch.__version__} | tv {torchvision.__version__} | "
      f"ultralytics {ultralytics.__version__} | cv2 {cv2.__version__}")

---
## 2 — Configuration
Edit the two variables below, then leave everything else alone.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)  # skips auth prompt on re-run

import os

# ─── EDIT THESE ───────────────────────────────────────────────────────────────
VIDEO_PATH    = "/content/drive/MyDrive/soccer/Denver vs. Virginia Full Match Replay | 2025 ACC Men's Soccer.mp4"
TARGET_JERSEY = 19
# ──────────────────────────────────────────────────────────────────────────────

OUTPUT_PATH   = f"/content/drive/MyDrive/soccer/highlights_jersey{TARGET_JERSEY}.mp4"
CONFIG_PATH   = os.path.join(REPO_DIR, "config.yaml")

assert os.path.exists(VIDEO_PATH), (
    f"Video not found: {VIDEO_PATH}\n"
    "Check the filename in Google Drive matches exactly (including punctuation)."
)
size_gb = os.path.getsize(VIDEO_PATH) / 1e9
print(f"Video : {os.path.basename(VIDEO_PATH)}")
print(f"Size  : {size_gb:.2f} GB")
print(f"Jersey: #{TARGET_JERSEY}")

---
## 3 — Trim a 10-minute gameplay clip
Skips any pre-game intro. Re-running this cell is instant if the clip already exists.

In [ ]:
import os

START_SEC    = 300    # 5:00 — adjust if there is a long intro
DURATION_SEC = 600   # 10 minutes of live play
CLIP_PATH    = f"/content/game_clip_jersey{TARGET_JERSEY}.mp4"

if not os.path.exists(CLIP_PATH):
    print("Trimming clip...")
    !ffmpeg -y -ss {START_SEC} -i "{VIDEO_PATH}" -t {DURATION_SEC} \
        -c copy "{CLIP_PATH}" -loglevel error
    print(f"Clip saved: {os.path.getsize(CLIP_PATH)/1e6:.1f} MB")
else:
    print(f"Clip already exists ({os.path.getsize(CLIP_PATH)/1e6:.1f} MB) — skipping trim")

VIDEO_PATH = CLIP_PATH

---
## 4 — Build model weights
Trains jersey CNN + LSTM scorer on synthetic stubs (~5 min on T4).
Skipped automatically if weights already exist.

In [ ]:
import os
os.chdir(REPO_DIR)

jersey_weights = os.path.join(REPO_DIR, "models/jersey_cnn.pt")
scorer_weights = os.path.join(REPO_DIR, "models/scorer_lstm.pt")

if not os.path.exists(jersey_weights) or not os.path.exists(scorer_weights):
    print("Weights missing — training now (~5 min on T4)...")
    !python scripts/download_soccernet.py --task stubs
    !python run_training.py --quick-test --stages jersey scorer --device cuda
else:
    print("Weights already present — skipping training")

# Hard stop if anything went wrong
assert os.path.exists(jersey_weights), "jersey_cnn.pt missing — check training output above"
assert os.path.exists(scorer_weights), "scorer_lstm.pt missing — check training output above"
print(f"jersey_cnn.pt  : {os.path.getsize(jersey_weights)/1e6:.1f} MB")
print(f"scorer_lstm.pt : {os.path.getsize(scorer_weights)/1e6:.1f} MB")
print("Weights ready")

---
## 5 — Run the full pipeline

In [ ]:
import os, sys, importlib
os.chdir(REPO_DIR)

# Force reimport so git pull fixes take effect without restarting the kernel
for mod in list(sys.modules.keys()):
    if mod.startswith("src") or mod == "main":
        del sys.modules[mod]

from main import run_pipeline

DEBUG_VIDEO = f"/content/debug_jersey{TARGET_JERSEY}.mp4"

result = run_pipeline(
    video_path=VIDEO_PATH,
    jersey_number=TARGET_JERSEY,
    output_path=OUTPUT_PATH,
    config_path=CONFIG_PATH,
    device="cuda",
    conf_override=0.25,
    highlight_thresh_override=0.40,
    frame_skip=1,
    clip_length_override=90,
    clip_stride_override=15,
    debug_video=DEBUG_VIDEO,
)
print(f"\nHighlight reel → {result}")

---
## 6 — Preview debug overlay
Green box = jersey #19. All tracked players labelled.

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os

PREVIEW = f"/content/debug_preview_jersey{TARGET_JERSEY}.mp4"
!ffmpeg -y -i "{DEBUG_VIDEO}" -vcodec libx264 -acodec aac \
    -vf "scale=960:-2" -crf 28 -preset fast "{PREVIEW}" -loglevel error

data = open(PREVIEW, 'rb').read()
b64  = b64encode(data).decode()
display(HTML(f'''
<h3>Debug overlay — jersey #{TARGET_JERSEY} highlighted in green</h3>
<video width="960" controls>
  <source src="data:video/mp4;base64,{b64}" type="video/mp4">
</video>
'''))

---
## 7 — Preview highlight reel

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os

if result and os.path.exists(result):
    REEL_PREVIEW = f"/content/reel_preview_jersey{TARGET_JERSEY}.mp4"
    !ffmpeg -y -i "{result}" -vcodec libx264 -acodec aac \
        -crf 26 -preset fast "{REEL_PREVIEW}" -loglevel error
    data = open(REEL_PREVIEW, 'rb').read()
    b64  = b64encode(data).decode()
    display(HTML(f'''
    <h3>Highlight reel — jersey #{TARGET_JERSEY}</h3>
    <video width="960" controls>
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    '''))
else:
    print(f"No highlights found for jersey #{TARGET_JERSEY}.")
    print("Options:")
    print("  1. Lower highlight_thresh_override to 0.30 in Cell 5 and re-run")
    print("  2. Run Cell 8 to check which jersey numbers the model actually sees")
    print("  3. Adjust START_SEC in Cell 3 to a busier stretch of play")

---
## 8 — Jersey frequency scan
Samples 300 frames and counts jersey reads. Use this to verify #19 is being
detected, or to discover which numbers the model reads most confidently.

In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
import os, sys
os.chdir(REPO_DIR)

from src.detector import PlayerDetector
from src.jersey_reader import JerseyReader

det = PlayerDetector(conf_threshold=0.25, device="cuda")
jr  = JerseyReader(
    model_path=os.path.join(REPO_DIR, "models/jersey_cnn.pt"),
    device="cuda"
)

cap     = cv2.VideoCapture(VIDEO_PATH)
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
counter = Counter()
indices = np.linspace(0, total - 1, min(300, max(1, total // 10)), dtype=int)

for fi in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
    ret, frame = cap.read()
    if not ret:
        continue
    for d in det.detect(frame):
        crop = det.crop_upper_body(frame, d)
        num, conf = jr.read(crop)
        if num is not None:
            counter[num] += 1
cap.release()

df = pd.DataFrame(counter.most_common(20), columns=["jersey", "sightings"])
print(df.to_string(index=False))

target_count = counter.get(TARGET_JERSEY, 0)
print(f"\n#{TARGET_JERSEY} sightings: {target_count} / {len(indices)} sampled frames")
if target_count == 0:
    print("Model did not read #19 in this sample — jersey CNN needs real SoccerNet training data.")
    print("The pipeline will still track based on detector + ByteTrack; jersey anchor won't fire.")

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(df["jersey"].astype(str), df["sightings"], color="steelblue")
# Highlight target jersey bar in green
for bar, jersey in zip(bars, df["jersey"]):
    if jersey == TARGET_JERSEY:
        bar.set_color("green")
ax.set_xlabel("Jersey number")
ax.set_ylabel("Sightings")
ax.set_title(f"Jersey frequency scan — #{TARGET_JERSEY} in green")
plt.tight_layout()
plt.savefig(f"/content/jersey_freq_jersey{TARGET_JERSEY}.png", dpi=150)
plt.show()